In [1]:
from platform import python_version
print(python_version())

3.11.14


In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as npmtd
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

import json
import requests
import pandas as pd

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.GDC_lib import GDC
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/config/all_lfc_cutoffs_TCGA-PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=False, verbose=False)
# print("\nEcho Parameters:")
# print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>>> Tumor


### GDC - no memory restriction to get all data available

In [5]:
gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
force=False
verbose=False

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)
prog_list.sort()

"; ".join(prog_list)

'ALCHEMIST; APOLLO; BEATAML1.0; CCDI; CCG; CDDP_EAGLE; CGCI; CMI; CPTAC; CTSP; EXCEPTIONAL_RESPONDERS; FM; HCMI; MATCH; MMRF; MP2PRT; NCICCR; OHSU; ORGANOID; RC; REBC; TARGET; TCGA; TRIO; VAREPOP; WCDT'

In [7]:
PROG_ID = 'TCGA'

df_psi = gdc.get_primary_sites(prog_id=PROG_ID, verbose=verbose)
print(len(df_psi))
df_psi.head(3)

33


,prog_id,gdc_project_id,disease_id,psi_id,primary_site,disease_context,cbioportal_study_id,mapping_status,recommended_mutation_source,notes,alternative_cbioportal_study_ids,review_notes
0,TCGA,TCGA-ACC,ACC,TCGA-ACC,Adrenal gland,Adrenocortical carcinoma,acc_tcga_pan_can_atlas_2018,formulaic_tcga_pan_can_atlas,GDC MAF first; cBioPortal only if mapping is validated,Validate with cBioPortal /api/studies before production.,NaN,NaN
1,TCGA,TCGA-BLCA,BLCA,TCGA-BLCA,Bladder,Bladder urothelial carcinoma,blca_tcga_pan_can_atlas_2018,formulaic_tcga_pan_can_atlas,GDC MAF first; cBioPortal only if mapping is validated,Validate with cBioPortal /api/studies before production.,NaN,NaN
2,TCGA,TCGA-BRCA,BRCA,TCGA-BRCA,Breast,Breast invasive carcinoma,brca_tcga_pan_can_atlas_2018,formulaic_tcga_pan_can_atlas,GDC MAF first; cBioPortal only if mapping is validated,Validate with cBioPortal /api/studies before production.,NaN,NaN


### Open primary cites from cbio

In [8]:
df_psi = gdc.open_primary_sites_cbio(verbose=True)

df_psi.columns

Table opened ((89, 12)) at '/home/flavio/uv/perturb_agent/data/gdc_to_cbioportal_study_mapping.tsv'


Index(['prog_id', 'gdc_project_id', 'disease_id', 'psi_id', 'primary_site', 'disease_context',
       'cbioportal_study_id', 'mapping_status', 'recommended_mutation_source', 'notes',
       'alternative_cbioportal_study_ids', 'review_notes'],
      dtype='object')

In [9]:
df_psi.head(3).T

,0,1,2
prog_id,TCGA,TCGA,TCGA
gdc_project_id,TCGA-ACC,TCGA-BLCA,TCGA-BRCA
disease_id,ACC,BLCA,BRCA
psi_id,TCGA-ACC,TCGA-BLCA,TCGA-BRCA
primary_site,Adrenal gland,Bladder,Breast
disease_context,Adrenocortical carcinoma,Bladder urothelial carcinoma,Breast invasive carcinoma
cbioportal_study_id,acc_tcga_pan_can_atlas_2018,blca_tcga_pan_can_atlas_2018,brca_tcga_pan_can_atlas_2018
mapping_status,formulaic_tcga_pan_can_atlas,formulaic_tcga_pan_can_atlas,formulaic_tcga_pan_can_atlas
recommended_mutation_source,GDC MAF first; cBioPortal only if mapping is validated,GDC MAF first; cBioPortal only if mapping is validated,GDC MAF first; cBioPortal only if mapping is validated
notes,Validate with cBioPortal /api/studies before production.,Validate with cBioPortal /api/studies before production.,Validate with cBioPortal /api/studies before production.


In [10]:
DISEASE_ID = 'ACC'
DISEASE_ID = 'PAAD'

df_psi = df_psi[ (df_psi.disease_id == DISEASE_ID) & (~pd.isnull(df_psi.primary_site)) & (~pd.isnull(df_psi.cbioportal_study_id)) ].copy()
dfa = df_psi.groupby(['prog_id', 'psi_id', 'disease_id', 'primary_site', 'gdc_project_id', 'cbioportal_study_id']).size().reset_index()

dfa[ ['prog_id', 'psi_id', 'disease_id', 'primary_site', 'gdc_project_id', 'cbioportal_study_id'] ]

,prog_id,psi_id,disease_id,primary_site,gdc_project_id,cbioportal_study_id
0,CCLE,CCLE-PAAD,PAAD,Pancreas,CCLE-PAAD,ccle_broad_2019
1,CPTAC,CPTAC-PAAD,PAAD,Pancreas,CPTAC-3,paad_cptac_2021
2,CPTAC,CPTAC-PAAD_GDC,PAAD,Pancreas,CPTAC-3,pancreas_cptac_gdc
3,TCGA,TCGA-PAAD,PAAD,Pancreas,TCGA-PAAD,paad_tcga_pan_can_atlas_2018


In [28]:
prog_id = 'CCLE'  # DepMap 
psi_id = 'CCLE-BRCA'

prog_id = 'CPTAC'
psi_id = 'CPTAC-BRCA'

prog_id = 'TCGA'
psi_id = 'TCGA-ACC'
psi_id = 'TCGA-BRCA'
psi_id = 'TCGA-BLCA'


df_psi = gdc.get_primary_sites(prog_id=prog_id, verbose=verbose)
df_psi.head(2)

Table opened ((89, 12)) at '/home/flavio/uv/perturb_agent/data/gdc_to_cbioportal_study_mapping.tsv'


,prog_id,gdc_project_id,disease_id,psi_id,primary_site,disease_context,cbioportal_study_id,mapping_status,recommended_mutation_source,notes,alternative_cbioportal_study_ids,review_notes
0,TCGA,TCGA-ACC,ACC,TCGA-ACC,Adrenal gland,Adrenocortical carcinoma,acc_tcga_pan_can_atlas_2018,formulaic_tcga_pan_can_atlas,GDC MAF first; cBioPortal only if mapping is validated,Validate with cBioPortal /api/studies before production.,NaN,NaN
1,TCGA,TCGA-BLCA,BLCA,TCGA-BLCA,Bladder,Bladder urothelial carcinoma,blca_tcga_pan_can_atlas_2018,formulaic_tcga_pan_can_atlas,GDC MAF first; cBioPortal only if mapping is validated,Validate with cBioPortal /api/studies before production.,NaN,NaN


In [29]:

gdc.set_primary_site(psi_id)

True

In [30]:
gdc.gdc_project_id, gdc.primary_site

('TCGA-BLCA', 'Bladder')

In [24]:
force=False
verbose=True

df_cases, df_subt, df_clin_demo = gdc.get_cases_and_subtypes(batch_size=200, do_filter=True, force=force, verbose=verbose)

print(df_cases.shape)
print(df_cases.primary_site.unique())
df_cases.head(3)

.......

👉 Returned 1098 / Total paginated 1098
Table saved ((1098, 26)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-BRCA/cases_for_TCGA-BRCA.tsv'
Table saved ((21, 6)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-BRCA/subtype_for_TCGA-BRCA.tsv'
..........................
--------------- end ------------


HTTPError: 414 Client Error: Request-URI Too Long for url: https://api.gdc.cancer.gov/cases?filters=%7B%22op%22%3A+%22in%22%2C+%22content%22%3A+%7B%22field%22%3A+%22case_id%22%2C+%22value%22%3A+%5B%2200807dae-9f4a-4fd1-aac2-82eb11bf2afb%22%2C+%2200b11ca8-8540-4a3d-b602-ec754b00230b%22%2C+%2202bbb632-0f7f-439d-b8f0-c86a06237424%22%2C+%2202bf5203-f9cd-4c5a-97b4-e5584dc22325%22%2C+%2203c143e0-d8a1-4d60-a4a3-df0501fc6b6e%22%2C+%22048549cf-d0a5-4743-a0a1-2004f7cc2b08%22%2C+%2205af737d-89aa-49b6-97b2-28b8ac8bd2cf%22%2C+%22077506ea-8f11-405d-bb8e-3c9924fe54d0%22%2C+%2208306ede-e74b-4be9-b768-7fd98647b6ac%22%2C+%220930e97a-72ea-4bd5-b2f0-50c76af695f3%22%2C+%2209d6e904-7184-4788-8a19-04f8c46cb13f%22%2C+%220adf59c6-581a-475d-a2f4-40aa40060b5b%22%2C+%220b0f9938-ce1e-4814-8821-0132d137eb9e%22%2C+%220bf6e772-1aa1-4f54-9a50-c1414e2f22f3%22%2C+%220cf0d67c-2f7f-42a6-94f3-407d0ec46c88%22%2C+%220d03055c-186b-418a-b614-e2035757bc3b%22%2C+%220d146997-cfdc-4587-888c-e876fc455c80%22%2C+%2210c829ec-fd66-49c4

In [15]:
gdc.df_clin_demo

,case_id,barcode_case,project_id,primary_site,disease_type,gender,age_at_diagnosis,race,ethnicity,vital_status,...,tumor_stage,ajcc_pathologic_stage,ajcc_pathologic_t,ajcc_pathologic_n,ajcc_pathologic_m,ajcc_clinical_stage,ajcc_clinical_t,ajcc_clinical_n,ajcc_clinical_m,classification_of_tumor
0,0304b12d-7640-4150-a581-2eea2b1f2ad5,TCGA-OR-A5LL,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,75.58,not reported,not reported,Dead,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,primary
1,0304b12d-7640-4150-a581-2eea2b1f2ad5,TCGA-OR-A5LL,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,75.58,not reported,not reported,Dead,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,metastasis
2,075dbfd0-9cf4-4877-884f-ae858902c79e,TCGA-OR-A5J7,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,31.32,white,hispanic or latino,Dead,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,metastasis
3,075dbfd0-9cf4-4877-884f-ae858902c79e,TCGA-OR-A5J7,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,31.32,white,hispanic or latino,Dead,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,metastasis
4,075dbfd0-9cf4-4877-884f-ae858902c79e,TCGA-OR-A5J7,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,31.32,white,hispanic or latino,Dead,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,recurrence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,fb504e55-1859-48ee-aec7-a3fcb082785e,TCGA-OR-A5LB,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,male,60.71,white,not hispanic or latino,Dead,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,primary
162,fb54458d-c373-46c2-841e-82663e13efaa,TCGA-OR-A5JM,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,25.62,white,not reported,Dead,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,primary
163,fb54458d-c373-46c2-841e-82663e13efaa,TCGA-OR-A5JM,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,25.62,white,not reported,Dead,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,metastasis
164,fc83ab27-929e-4ad4-ba18-abb0694f1e43,TCGA-OR-A5LJ,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,55.21,white,not hispanic or latino,Dead,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,metastasis


In [27]:
df_clin_demo = gdc.get_gdc_clin_demo_data(df_cases.case_id, force=True, batch_size=50, verbose=True)
df_clin_demo

.........
--------------- end ------------
Batch 1: 90 requested, 90 returned
Table saved ((166, 29)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-BRCA/clinical_and_demographics_for_TCGA-BRCA.tsv'


,case_id,barcode_case,project_id,primary_site,disease_type,gender,age_at_diagnosis,race,ethnicity,vital_status,...,tumor_stage,ajcc_pathologic_stage,ajcc_pathologic_t,ajcc_pathologic_n,ajcc_pathologic_m,ajcc_clinical_stage,ajcc_clinical_t,ajcc_clinical_n,ajcc_clinical_m,classification_of_tumor
0,0304b12d-7640-4150-a581-2eea2b1f2ad5,TCGA-OR-A5LL,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,75.58,not reported,not reported,Dead,...,None,None,None,None,None,None,None,None,None,primary
1,0304b12d-7640-4150-a581-2eea2b1f2ad5,TCGA-OR-A5LL,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,75.58,not reported,not reported,Dead,...,None,None,None,None,None,None,None,None,None,metastasis
2,075dbfd0-9cf4-4877-884f-ae858902c79e,TCGA-OR-A5J7,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,31.32,white,hispanic or latino,Dead,...,None,None,None,None,None,None,None,None,None,metastasis
3,075dbfd0-9cf4-4877-884f-ae858902c79e,TCGA-OR-A5J7,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,31.32,white,hispanic or latino,Dead,...,None,None,None,None,None,None,None,None,None,metastasis
4,075dbfd0-9cf4-4877-884f-ae858902c79e,TCGA-OR-A5J7,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,31.32,white,hispanic or latino,Dead,...,None,None,None,None,None,None,None,None,None,recurrence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,fb504e55-1859-48ee-aec7-a3fcb082785e,TCGA-OR-A5LB,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,male,60.71,white,not hispanic or latino,Dead,...,None,None,None,None,None,None,None,None,None,primary
162,fb54458d-c373-46c2-841e-82663e13efaa,TCGA-OR-A5JM,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,25.62,white,not reported,Dead,...,None,None,None,None,None,None,None,None,None,primary
163,fb54458d-c373-46c2-841e-82663e13efaa,TCGA-OR-A5JM,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,25.62,white,not reported,Dead,...,None,None,None,None,None,None,None,None,None,metastasis
164,fc83ab27-929e-4ad4-ba18-abb0694f1e43,TCGA-OR-A5LJ,TCGA-ACC,Adrenal gland,Adenomas and Adenocarcinomas,female,55.21,white,not hispanic or latino,Dead,...,None,None,None,None,None,None,None,None,None,metastasis


In [17]:
df_subt

,psi_id,subtype_global,tumor_class,subtype_tissue,stage,n
0,TCGA-ACC,Adrenal cortical carcinoma,adrenal_cortical_carcinoma,Adrenal cortical carcinoma,NaN,87
1,TCGA-ACC,Adrenal cortical carcinoma,adrenal_cortical_carcinoma,Adrenal cortical carcinoma,unknown,2
2,TCGA-ACC,Adrenal cortical carcinoma,sarcoma,Adrenal cortical carcinoma,NaN,1


In [18]:
for isubt, row in df_subt.iterrows():
    subtype_global = row.subtype_global
    tumor_class = row.tumor_class
    subtype_tissue = row.subtype_tissue

    df_samples = gdc.get_samples_for_subtypes(
        subtype_global=subtype_global,
        tumor_class=tumor_class,
        subtype_tissue=subtype_tissue,
        batch_size=200,
        force=False,
        verbose=verbose,
    )
    print(f"{isubt}) {gdc.s_case}")

    if df_samples.empty:
        print(f"No samples found for PSI_ID: {psi_id} subtype: {subtype_global} tumor_class: {tumor_class} subtype_tissue: {subtype_tissue}")
    else:
        print(f"There are {len(df_samples)} samples for PSI_ID: {psi_id} subtype: {subtype_global} tumor_class: {tumor_class} subtype_tissue: {subtype_tissue}")


Table opened ((92, 26)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-ACC/cases_for_TCGA-ACC.tsv'
Table opened ((166, 29)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-ACC/clinical_and_demographics_for_TCGA-ACC.tsv'
Table opened ((4481, 14)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-ACC/samples/samples_for_TCGA-ACC_Adrenal_gland_subtype_Adrenal_cortical_carcinoma_tumor_adrenal-cortical-carcinoma_tissue_Adrenal_cortical_carcinoma.tsv'
0) TCGA-ACC_Adrenal_gland_subtype_Adrenal_cortical_carcinoma_tumor_adrenal-cortical-carcinoma_tissue_Adrenal_cortical_carcinoma
There are 4481 samples for PSI_ID: TCGA-ACC subtype: Adrenal cortical carcinoma tumor_class: adrenal_cortical_carcinoma subtype_tissue: Adrenal cortical carcinoma
Table opened ((92, 26)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-ACC/cases_for_TCGA-ACC.tsv'
Table opened ((166, 29)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-ACC/clinical_and_demographics_for_TCGA-ACC.tsv'
Table opened ((4481, 14)) at '/hom